# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [1]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.linear_model import LogisticRegression

In [3]:

# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv('titanic_data_updated.csv')


# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
735,736,no,third,"Williams, Mr. Leslie",male,28.5,0,0,54636,16.1000,NaN,S
73,74,no,third,"Chronopoulos, Mr. Apostolos",male,26.0,1,0,2680,14.4542,NaN,C
711,712,no,first,"Klaber, Mr. Herman",male,NaN,0,0,113028,26.5500,C124,S
155,156,no,first,"Williams, Mr. Charles Duane",male,51.0,0,1,PC 17597,61.3792,NaN,C
743,744,no,third,"McNamee, Mr. Neal",male,24.0,1,0,376566,16.1000,NaN,S


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [4]:
# YOUR CODE HERE
df["Family_Size"] = df["SibSp"] + df['Parch']+ 1

new_order = [
    'PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age',
    'SibSp', 'Parch', 'Family_Size',  # <--- Ekhane SibSp ar Parch er pore niye asa holo
    'Ticket', 'Fare', 'Cabin', 'Embarked']
df = df[new_order]


df['Cabin']= df['Cabin'].fillna('Missing')
df['Embarked']= df['Embarked'].fillna('Unknown')


df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Family_Size,Ticket,Fare,Cabin,Embarked
478,479,no,third,"Karlsson, Mr. Nils August",male,22.0,0,0,1,350060,7.5208,Missing,S
545,546,no,first,"Nicholson, Mr. Arthur Ernest",male,64.0,0,0,1,693,26.0000,Missing,S
760,761,no,third,"Garfirth, Mr. John",male,NaN,0,0,1,358585,14.5000,Missing,S
576,577,yes,second,"Garside, Miss. Ethel",female,34.0,0,0,1,243880,13.0000,Missing,S
825,826,no,third,"Flynn, Mr. John",male,NaN,0,0,1,368323,6.9500,Missing,Q


In [5]:
df['Cabin'].info()


<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
891 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [8]:
df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)


df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [11]:
# YOUR CODE HERE
X = df.drop('Survived', axis=1)
y = df['Survived']

print(X.shape)
print(y.shape)

(891, 13)
(891,)


---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [13]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(
                                  X, y,
                                  test_size=0.2, random_state=42, stratify=y
                                  )

print(X_train.shape)
print(X_test.shape)

(712, 13)
(179, 13)


---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [14]:
# YOUR CODE HERE
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [16]:
# YOUR CODE HERE
# pipeline Numirical

p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)
p1
p2

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', MinMaxScaler())])

---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [18]:
# YOUR CODE HERE
categories = [['third','second','first']]

# pipeline
#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())]
        )

---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [19]:
# YOUR CODE HERE
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [22]:
# YOUR CODE HERE

preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'

)





In [23]:
lr_model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',LogisticRegression(class_weight='balanced',max_iter=1000))
    ]
)
lr_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline_1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_Size']),
                                                 ('pipeline_3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strateg...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [24]:
lr_model.fit(X_train,y_train)

lr_model['model'].coef_
lr_model['model'].intercept_
lr_model['model'].classes_

array([0, 1])

In [25]:
# YOUR CODE HERE
y_pred = lr_model.predict(X_test)

lr_model.predict_proba(X_test)

array([[0.85526115, 0.14473885],
       [0.92585208, 0.07414792],
       [0.7666301 , 0.2333699 ],
       [0.92801033, 0.07198967],
       [0.37660805, 0.62339195],
       [0.44238046, 0.55761954],
       [0.25270683, 0.74729317],
       [0.59796523, 0.40203477],
       [0.47968315, 0.52031685],
       [0.78364905, 0.21635095],
       [0.8621186 , 0.1378814 ],
       [0.81680817, 0.18319183],
       [0.33890802, 0.66109198],
       [0.70246038, 0.29753962],
       [0.14975119, 0.85024881],
       [0.83774933, 0.16225067],
       [0.46573884, 0.53426116],
       [0.86481338, 0.13518662],
       [0.79276925, 0.20723075],
       [0.23744971, 0.76255029],
       [0.86481338, 0.13518662],
       [0.1991492 , 0.8008508 ],
       [0.86652566, 0.13347434],
       [0.46127614, 0.53872386],
       [0.86184692, 0.13815308],
       [0.02715049, 0.97284951],
       [0.67634473, 0.32365527],
       [0.6359975 , 0.3640025 ],
       [0.81348342, 0.18651658],
       [0.80965131, 0.19034869],
       [0.

In [26]:
from sklearn.metrics import accuracy_score,precision_score,recall_score

In [27]:

accuracy = accuracy_score(y_test,y_pred)
print(accuracy)
precision = precision_score(y_test,y_pred)
print(precision)
recall = recall_score(y_test,y_pred)
print(recall)

0.7653631284916201
0.6753246753246753
0.7536231884057971
